In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# How many pages?

In [3]:
len(documents)

72

# Index search

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [10]:
results = index.search(
    query = "How does the agentic loop keep calling the model until it stops?",
    num_results = 4
)

[result["filename"] for result in results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md']

# RAG using index

In [16]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

In [20]:
class LessonPageRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results = num_results
        )

    def build_context(self, search_results):
        lines = []

        for document in search_results:
            lines.append(document['content'])

        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return {"answer": response.output_text, "input_tokens": response.usage.input_tokens}

In [21]:
lessonrag = LessonPageRAG(
    index = index,
    llm_client=openai_client
)

In [22]:
response = lessonrag.rag("How does the agentic loop keep calling the model until it stops?")

In [23]:
response

{'answer': 'It keeps calling the model in a `while True` loop.\n\nAfter each model response, the code checks whether there were any `function_call` items:\n\n- if yes, it runs the tool, appends the tool result to `messages`, and loops again\n- if no, it breaks out of the loop\n\nSo the stop condition is: **the model returns a response with no function calls**.',
 'input_tokens': 7036}

# Chunking

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
len(chunks)

295

# RAG with chunking

In [28]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [29]:
lessonrag_chunk = LessonPageRAG(
    index = chunk_index,
    llm_client=openai_client
)

In [30]:
response_chunk = lessonrag_chunk.rag("How does the agentic loop keep calling the model until it stops?")

In [31]:
response_chunk

{'answer': 'It keeps looping with a `while True` and checks whether the model made any `function_call`s on that turn.\n\n- If there is at least one `function_call`, the code runs the tool, appends the result, and continues.\n- If there are no `function_call`s, it breaks out of the loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, the loop stops when the model returns a final answer with no more tool calls.',
 'input_tokens': 2221}

# Turn into an agent

In [43]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [49]:
def search(query: str) -> dict[str, str]:
    """
    This tool searches the course page database for pages with content that matches the query
    """
    return index.search(
        query,
        num_results = 5
    )

In [50]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [51]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'This tool searches the course page database for pages with content that matches the query',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [52]:
INSTRUCTIONS = '''
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
'''

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [53]:
runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nAnswer the student's question using the search tool.\nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG difference agentic loop"}', call_id='call_JcFYsRkqVs2iVWkZ11O2koBR', name='search', type='function_call', id='fc_0977b7cfe9d23f3d006a3ddefc3980819da2ae9f285a62d5f7', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop retrieve act reflect course"}', call_id='call_aXhSB8wn3dCuFVbi1t4Jiwmo', name='search', type='function_call', id='fc_0977b7cfe9d23f3d006a3ddefc399c819d8ee8de110c1df564', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG versus agentic